# Core-satellite trend portfolio for a family office
**2270 Financial Modeling, Group Assignment** (Group: _TODO_, Members: _TODO_)

Streamlit app: _TODO URL_ · GitHub repository: https://github.com/nilscuypers18-ops/FM-Group-Project

## Setup instructions
1. Open this notebook in a fresh Google Colab session (colab.research.google.com).
2. The first code cell downloads `strategy.py` from our GitHub repository (`main` branch).
3. Run all cells (Runtime > Run all). Data is downloaded live from Yahoo Finance through the `yfinance` API. No data files are needed.

All calculations live in `strategy.py`, so this notebook, the Streamlit app and the prospectus use exactly the same code.

In [ ]:
!pip install -q yfinance

import os, urllib.request
REPO_RAW_URL = "https://raw.githubusercontent.com/nilscuypers18-ops/FM-Group-Project/main"
if not os.path.exists("strategy.py"):
    urllib.request.urlretrieve(f"{REPO_RAW_URL}/strategy.py", "strategy.py")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import strategy as st

pd.options.display.float_format = "{:.4f}".format

## 1. Parameters and sample period
Start and end dates are set explicitly here. Prices are downloaded from February 2006 (DBC starts trading in February 2006) so that the 10-month moving average and the 12-month volatility are available before the first backtest month, January 2007.

In [ ]:
DOWNLOAD_START = "2006-02-01"
BACKTEST_START = "2007-01-31"   # first month with a portfolio return
BACKTEST_END   = "2026-10-31"   # last month with a portfolio return

CORE_WEIGHT = 0.60   # AGG
MA_MONTHS   = 10     # trend filter
COST_BPS    = 10     # transaction cost per unit of turnover

## 2. Data
* **Source:** Yahoo Finance via `yfinance`, adjusted close prices (dividends and splits reinvested), USD.
* **Assets:** AGG (US aggregate bonds), SPY (US equity), GLD (gold), DBC (commodities). Cash: 3-month US T-bill yield `^IRX`.
* **Frequency:** daily prices converted to month-end (last trading day of each month).
* **T-bill return:** monthly return in month t = `^IRX` yield at the end of month t-1 / 100 / 12.

In [ ]:
prices, rf = st.download_data(DOWNLOAD_START, BACKTEST_END)
print(prices.index[0].date(), "to", prices.index[-1].date(), "|", len(prices), "month-ends")
prices.tail()

## 3. Signals and weights
At each month-end t:
1. An asset is **on** if its price is above its 10-month moving average.
2. The 40% satellite budget is split across SPY, GLD and DBC in proportion to 1 / (12-month volatility).
3. The share of an asset that is **off** goes to T-bills. AGG stays at 60%.

These weights are held during month t+1, so every decision uses only information available at the time.

In [ ]:
weights = st.strategy_weights(prices, CORE_WEIGHT, MA_MONTHS)
weights.loc[BACKTEST_START:].plot.area(figsize=(10, 4), title="Strategy weights (decided at month-end, held next month)")
plt.ylabel("Weight"); plt.margins(0, 0); plt.show()
weights.tail()

## 4. Backtest
Comparison portfolios:
* **60/40 SPY/AGG** benchmark, rebalanced monthly.
* **Buy & hold (same assets):** 60% AGG and 13.3% each in SPY, GLD and DBC, bought once in January 2007 and never rebalanced.

All portfolios pay 10 bps per unit of turnover, including the initial purchase.

In [ ]:
results, rf_bt, w_strat, res_strat = st.run_all(prices, rf, CORE_WEIGHT, MA_MONTHS, COST_BPS,
                                              BACKTEST_START, BACKTEST_END)
cum = st.cumulative(results)
cum.plot(figsize=(10, 5), title="Cumulative performance (start = 1, net of costs)")
plt.ylabel("Value of 1 invested"); plt.grid(alpha=0.3); plt.show()

## 5. Performance measures
Calculation assumptions:
* Monthly returns, net of transaction costs. Risk-free rate = 3-month T-bill.
* **Annualized average return** = 12 × mean monthly return. CAGR is shown as well.
* **Volatility** = √12 × standard deviation of monthly returns.
* **Sharpe ratio** = annualized mean excess return over T-bills / annualized volatility of excess returns.
* **Beta and alpha:** OLS regression of the portfolio's monthly excess return on the 60/40 benchmark's monthly excess return. Beta is **not** annualized; alpha = 12 × monthly intercept.
* **Information ratio** = annualized mean active return (portfolio − benchmark) / annualized tracking error.

In [ ]:
table = st.performance_table(results, rf_bt)
table.style.format({c: "{:.2%}" for c in ["Ann. return", "CAGR", "Volatility", "Alpha (ann.)",
                                         "Tracking error", "Max drawdown"]}
                   | {c: "{:.2f}" for c in ["Sharpe", "Beta", "Information ratio"]})

## 6. Implementation: turnover and costs

In [ ]:
print(f"Average monthly turnover (excl. first month): {res_strat['turnover'].iloc[1:].mean():.2%}")
print(f"Total costs paid (sum of monthly cost drag): {res_strat['cost'].sum():.2%}")
print(f"Share of months with each asset 'on':")
print((w_strat.loc[BACKTEST_START:, st.SATELLITE] > 0).mean().round(2))